In [2]:
from pyspark.sql import SparkSession
import numpy as np
import pandas as pd

spark = (
    SparkSession.builder.appName("MAST30034 Project 2")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/09 20:06:28 WARN Utils: Your hostname, PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/09 20:06:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/09 20:06:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/09 20:06:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/09/09 20:06:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


4.0.1


In [4]:
all_homes = pd.read_csv('data/all_homes.csv', skiprows=1)
all_homes.rename(columns={'All properties': 'region'}, inplace=True)

one_flat = pd.read_csv('data/one_bed_flat.csv', skiprows=1)
one_flat.rename(columns={'1 bedroom flat': 'region'}, inplace=True)

two_flat = pd.read_csv('data/two_bed_flat.csv', skiprows=1)
two_flat.rename(columns={'2 bedroom flat': 'region'}, inplace=True)

three_flat = pd.read_csv('data/three_bed_flat.csv', skiprows=1)
three_flat.rename(columns={'3 bedroom flat': 'region'}, inplace=True)

four_house = pd.read_csv('data/four_bed_house.csv', skiprows=1)
four_house.rename(columns={'4 bedroom house': 'region'}, inplace=True)

three_house = pd.read_csv('data/three_bed_house.csv', skiprows=1)
three_house.rename(columns={'3 bedroom house': 'region'}, inplace=True)

two_house = pd.read_csv('data/two_bed_house.csv', skiprows=1)
two_house.rename(columns={'2 bedroom house': 'region'}, inplace=True)


def reformat(df_bad):
    df_bad.rename(columns={'Unnamed: 1': 'suburb'}, inplace=True)
    df_bad['region'] = df_bad['region'].ffill()
    cols = df_bad.columns.to_series().replace(r'^Unnamed:.*', np.nan, regex=True)
    cols = cols.ffill()
    df_bad.columns = cols

    new_cols = []

    for i, col in enumerate(df_bad.columns[2:]):
        if i % 2 == 0:
            new_cols.append(f"{col} Count")
        else:
            new_cols.append(f"{col} Median")

    df_bad.columns = list(df_bad.columns[:2]) + new_cols
    df_bad = df_bad.drop(df_bad.index[0])

    df_long = df_bad.melt(id_vars=['region','suburb'], 
                            var_name='date_measure', 
                            value_name='value')

    df_long[['date','measure']] = df_long['date_measure'].str.rsplit(' ', n=1, expand=True)

    df_long['value'] = df_long['value'].replace('-', np.nan)
    df_long['value'] = df_long['value'].replace('[\$,]', '', regex=True)
    df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')

    df_final = df_long.pivot(index=['region','suburb','date'],
                            columns='measure',
                            values='value').reset_index()

    df_final = df_final.rename(columns={'Count':'count', 'Median':'median'})
    df_final.columns.name = None

    df_final['date'] = pd.to_datetime(df_final['date'], format='%b %Y')
    return df_final

all_reform = reformat(all_homes)
all_reform.to_csv("data_reformatted/all_reform.csv", index=False)

one_flat_remform = reformat(one_flat)
one_flat_remform.to_csv("data_reformatted/one_flat_remform.csv", index=False)

two_flat_remform = reformat(two_flat)
two_flat_remform.to_csv("data_reformatted/two_flat_remform.csv", index=False)

three_flat_remform = reformat(three_flat)
three_flat_remform.to_csv("data_reformatted/three_flat_remform.csv", index=False)

two_house_reform = reformat(two_house)
two_house_reform.to_csv("data_reformatted/two_house_reform.csv", index=False)

three_house_reform = reformat(three_house)
three_house_reform.to_csv("data_reformatted/three_house_reform.csv", index=False)

four_house_reform = reformat(four_house)
four_house_reform.to_csv("data_reformatted/four_house_reform.csv", index=False)

<>:48: SyntaxWarning: invalid escape sequence '\$'
<>:48: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_10096/2288263884.py:48: SyntaxWarning: invalid escape sequence '\$'
  df_long['value'] = df_long['value'].replace('[\$,]', '', regex=True)
